# Capstone build --- Chapter 3: Reasoning Traces

The loop in Chapter~1 recorded what the agent did --- the action and the observation at each step. Chapter~3 adds a record of what the agent may conclude: a reasoning trace in which every entry is typed, carries a trust level and points at its evidence. The capstone builds this trace with the `Scratchpad`, and enforces one rule on it before any customer-facing output leaves the system --- every claim must carry evidence.

## A typed reasoning entry

The `Scratchpad` is an append-only collection of typed entries. An observation records something a tool reported and names its source; a claim asserts something the reply may depend on and must carry evidence; an assumption records something inferred without textual support, which is precisely what must not be allowed to pass for a fact. Each entry carries a trust level.

In [ ]:
from agentlab.reasoning import Scratchpad, TrustLevel

pad = Scratchpad()
pad.add_observation(
    'message classified as complaint',
    source='classify_complaint',
    evidence='classifier confidence=1.00',
    trust=TrustLevel.MEDIUM,
)
pad.add_claim(
    'issue is unauthorized_fee',
    evidence='extract_facts grounded to policy entity [overdraft_fee]',
    trust=TrustLevel.HIGH,
)
pad.add_assumption('urgency high, sentiment negative')
print(pad.render_table(as_string=True))

The trace separates what was observed, what is claimed and what was merely assumed. The assumption about urgency and sentiment is recorded honestly as an assumption: it was inferred without a citation, so it can never pass a check for evidence it has not earned.

## The evidence check

The discipline the capstone enforces is that no claim reaches an output step without evidence. `assert_all_claims_have_evidence` is the structural check the harness runs; a claim added without an evidence pointer makes it raise, and the agent escalates to a human rather than letting an ungrounded statement through.

In [ ]:
pad.assert_all_claims_have_evidence()   # passes: the only claim has evidence
print('all claims grounded:', not pad.unsupported_claims())

pad.add_claim('the fee was a bank error')   # no evidence pointer
try:
    pad.assert_all_claims_have_evidence()
except AssertionError as exc:
    print('blocked before output:', exc)
    print('unsupported:', [e.text for e in pad.unsupported_claims()])

In the capstone this trace is not written by hand. The `ComplaintAgent` reconstructs the scratchpad from its tool results as a pure function of the trajectory: the same trajectory always yields the same trace, so it replays from the audit log without rerunning the agent. Each tool output enters as a typed entry --- the classifier result as an observation, a grounded issue as a high-trust claim, an implicated regulation as a highest-trust claim checked against the graph --- and the evidence check runs before the draft step. Chapter~4 formalizes the task, state and action types this trace hangs on; the governance gates of Chapter~6 and the escalation path of Chapter~13 are what act on a failed check.